# Agréger et comparer

Pour comparer des catégories, calculer un total par catégorie avant de construire les barres.

**Question métier :** Quelle région génère le plus de ventes dans ce jeu de données ?

Les données sont montées en lecture seule. Enregistrez vos figures dans `/workspace/resultats`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = '/workspace/data/sample_-_superstore.xls'
orders = pd.read_excel(DATA, sheet_name='Orders', engine='xlrd')
orders['Order Date'] = pd.to_datetime(orders['Order Date'])

## Notion 1 — Comparer les ventes par catégorie


### Étape 1 — Calculer les ventes

groupby rassemble les lignes d’une catégorie ; sum additionne les ventes.

In [ ]:
ventes = orders.groupby('Category')['Sales'].sum()
print(ventes)

### Étape 2 — Construire le graphique

index contient les catégories ; values contient les montants correspondants.

In [ ]:
ventes = ventes.sort_values()
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(ventes.index, ventes.values)
ax.set(title='Ventes par catégorie', xlabel='Ventes (unités monétaires)')
plt.show()

### Étape 3 — Interpréter

Technology réalise le total de ventes le plus élevé, environ 839 893 unités monétaires. Le graphique décrit le chiffre d’affaires, pas la rentabilité.

### Exercice 1 — Comparer les ventes par région

Quelle région réalise le plus de ventes ? Reproduisez le calcul et construisez un graphique horizontal trié.

**À rendre :** Le graphique, son titre, l’unité sur l’axe et une phrase qui distingue constat et explication.

In [ ]:
ventes_region = orders.groupby('Region')['Sales'].sum().sort_values()
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(ventes_region.index, ventes_region.values)
ax.set(title='Ventes par région', xlabel='Ventes (unités monétaires)')
plt.show()

### Mise en commun

West arrive en tête avec environ 739 814 unités monétaires. Cette comparaison n’indique pas pourquoi les régions diffèrent.

## Notion 2 — Calculer une part du total


### Étape 1 — Calculer le dénominateur

Le total de la colonne Sales sert de référence commune aux catégories.

In [ ]:
ventes = orders.groupby('Category')['Sales'].sum()
total = ventes.sum()
print(total)

### Étape 2 — Passer en pourcentage

Chaque part est le total de la catégorie divisé par le total général, multiplié par 100.

In [ ]:
part = ventes / total * 100
part = part.sort_values()
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(part.index, part.values)
ax.set(title='Part des ventes par catégorie', xlabel='Part des ventes (%)')
plt.show()

### Étape 3 — Interpréter

Les parts des catégories totalisent 100 %. Technology pèse environ 36,1 % des ventes du fichier.

### Exercice 2 — Calculer les parts régionales

Calculez la part de chaque région dans les ventes totales et représentez les quatre parts.

**À rendre :** Quatre parts, un graphique et une phrase vérifiant leur somme.

In [ ]:
ventes_region = orders.groupby('Region')['Sales'].sum()
part_region = ventes_region / ventes_region.sum() * 100
part_region = part_region.sort_values()
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(part_region.index, part_region.values)
ax.set(title='Part des ventes par région', xlabel='Part des ventes (%)')
plt.show()
print(part_region.sum())

### Mise en commun

West pèse environ 31,8 % des ventes, devant East (29,7 %), Central (21,6 %) et South (16,8 %). Les parts font 100 % à l’arrondi près.

## Notion 3 — Comparer le poids des catégories dans ventes et profit


### Étape 1 — Calculer la part de chaque total

Réutilisez le calcul d’une part du total pour deux mesures différentes : les ventes et le profit.

In [ ]:
bilan = orders.groupby("Category")[["Sales", "Profit"]].sum()
parts = pd.DataFrame(index=bilan.index)
parts["Sales"] = bilan["Sales"] / bilan["Sales"].sum() * 100
parts["Profit"] = bilan["Profit"] / bilan["Profit"].sum() * 100
print(parts.round(1))

### Étape 2 — Tracer les deux parts

Chaque série totalise 100 %. Les barres groupées comparent le poids relatif de chaque catégorie.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
y = range(len(parts))
ax.barh([i-.18 for i in y], parts['Sales'], height=.36, label='Part des ventes')
ax.barh([i+.18 for i in y], parts['Profit'], height=.36, label='Part du profit')
ax.set_yticks(list(y), parts.index)
ax.set(title='Part de chaque catégorie dans les totaux', xlabel='Part (%)')
ax.legend()
plt.show()

### Étape 3 — Interpréter

Technology pèse environ 36,1 % des ventes et 50,1 % du profit total. Les deux séries sont exprimées en parts de leur propre total, pas en montants directement comparables.

### Exercice 3 — Comparer les contributions régionales

Quelles régions pèsent davantage dans le profit total que dans les ventes totales ? Représentez l’écart entre les deux parts pour chaque région.

**À rendre :** Un graphique des quatre écarts en points de pourcentage et une phrase qui identifie les régions concernées.

In [ ]:
bilan_region = orders.groupby("Region")[["Sales", "Profit"]].sum()
part_ventes = bilan_region["Sales"] / bilan_region["Sales"].sum() * 100
part_profit = bilan_region["Profit"] / bilan_region["Profit"].sum() * 100
ecarts = (part_profit - part_ventes).sort_values()
print(ecarts)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ecarts.index, ecarts.values, color="#275c47")
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Part du profit moins part des ventes", xlabel="Écart (points de pourcentage)")
plt.tight_layout()
plt.show()

### Mise en commun

East (+2,72 points) et West (+6,11 points) pèsent davantage dans le profit que dans les ventes. Les écarts sont négatifs pour Central (−7,99 points) et South (−0,84 point). Une différence entre deux pourcentages s’exprime en points.

## Notion 4 — Repérer les sous-catégories les plus vendues


### Étape 1 — Agréger et sélectionner

nlargest conserve les huit sous-catégories aux ventes les plus élevées.

In [ ]:
ventes_sous_cat = orders.groupby('Sub-Category')['Sales'].sum()
top8 = ventes_sous_cat.nlargest(8).sort_values()
print(top8)

### Étape 2 — Représenter un classement lisible

Les barres horizontales affichent les noms longs sans les incliner.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(top8.index, top8.values)
ax.set(title='Huit premières sous-catégories', xlabel='Ventes (unités monétaires)')
plt.show()

### Étape 3 — Interpréter

Chairs et Phones sont les sous-catégories les plus vendues parmi les huit affichées. Le classement ne dit pas lesquelles sont les plus profitables.

### Exercice 4 — Construire le classement des clients

Quels sont les cinq clients, identifiés par Customer ID, qui réalisent le plus de ventes ? Représentez leur classement.

**À rendre :** Un graphique trié des cinq clients, leurs montants de ventes et une phrase de constat.

In [ ]:
ventes_clients = orders.groupby("Customer ID")["Sales"].sum()
top5 = ventes_clients.nlargest(5).sort_values()
print(top5)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top5.index, top5.values, color="#275c47")
ax.set(title="Les cinq clients réalisant le plus de ventes", xlabel="Ventes (unités monétaires)")
plt.tight_layout()
plt.show()

### Mise en commun

Les cinq premiers clients sont SM-20320 (25 043,05), TC-20980 (19 052,22), RB-19360 (15 117,34), TA-21385 (14 595,62) et AB-10105 (14 473,57). Le classement porte sur les ventes cumulées par client dans le fichier.